In [27]:
import pandas as pd 


In [28]:

def get_metrics_by_id(exp_path, cluster_id, gamma, ss_ids):
    gamma_no_point = str(gamma).replace(".", "")
    exp = "ss_"
    exp_name = exp_path.split("/")[-1]
    if "ra" in exp_name:
        exp += "ra_"
    else:
        exp += "sa_"

    if "er" in exp:
        exp += "er"
    else:
        exp += "sj"


    clustering_metrics_file = "clustering_metrics_" + str(gamma_no_point) + ".csv"
    leiden_clustering_file = "leidenClustering_" + str(gamma) + ".csv"
    df_metrics = pd.read_csv(f"{exp_path}/output/{clustering_metrics_file}")
    df_leiden = pd.read_csv(f"{exp_path}/output/{leiden_clustering_file}")


    df_leiden = df_leiden[df_leiden["cluster_id"] == cluster_id]
    df_leiden = df_leiden[df_leiden["node_id"].isin(ss_ids)]
    num_ss = df_leiden.shape[0]

    df_metrics = df_metrics[df_metrics["cluster_id"] == cluster_id]
    df_metrics.drop(columns=["intra_edges", "boundary_edges", "normalized_density"], inplace=True)
    df_metrics["experiment"] = exp
    df_metrics["ss_count"] = num_ss

    dict_metrics = df_metrics.to_dict(orient="records")
    return dict_metrics
    
    

In [30]:
gamma = 0.001
exp_path = "data/er/7_7_ecc7_ss_ra_new_pubmed_er"
cluster_id = 0
ss_ids = [491534, 491535, 491536]

dict_metrics = get_metrics_by_id(exp_path, cluster_id, gamma, ss_ids)
print(dict_metrics)

[{'cluster_id': 0, 'size': 18230, 'mincut': 19, 'avg_recency_w': 0.2839, 'avg_pa_w': 0.4189, 'avg_fitness_w': 0.2972, 'avg_fitness': 74.9682, 'seeds': 0, 'agents': 18230, 'experiment': 'ss_ra_sj', 'ss_count': 2}]


In [ ]:
edgelist_path = "data/er/7_7_ecc7_ss_ra_new_pubmed_er/output/output.edgelist"
aux_path = "data/er/7_7_ecc7_ss_ra_new_pubmed_er/output/output.aux"
df_edge_list = pd.read_csv(edgelist_path)
df_aux = pd.read_csv(aux_path, low_memory=False)

ss_ids = df_aux[df_aux["fit_peak_value"] > 1000]["node_id"].to_list()
for id_ in ss_ids:

    citings_ids = df_edge_list[df_edge_list["target"] == id_]["#source"].tolist()

    filtered_aux = df_aux[df_aux["node_id"].isin(citings_ids)]

    average_pa_weight_net = filtered_aux.groupby("year")["pa_weight"].mean().reset_index()
    average_rec_weight_net = filtered_aux.groupby("year")["rec_weight"].mean().reset_index()
    average_fit_weight_net = filtered_aux.groupby("year")["fit_weight"].mean().reset_index()

    ss_fit = df_aux[df_aux["node_id"] == id_]["fit_peak_value"].values[0]
    average_pa_weight_net.to_csv(f"avg_pa_net_{ss_fit}.csv", index=False)
    average_rec_weight_net.to_csv(f"avg_rec_net_{ss_fit}.csv", index=False)
    average_fit_weight_net.to_csv(f"avg_fit_net_{ss_fit}.csv", index=False)
    


In [ ]:
leidenClustering_path = "data/er/7_7_ecc7_ss_ra_new_pubmed_er/output/leidenClustering_0.01.csv"
df_leiden = pd.read_csv(leidenClustering_path)
df_aux = pd.read_csv(aux_path, low_memory=False)

ss_ids = df_aux[df_aux["fit_peak_value"] > 1000]["node_id"].to_list()
for id in ss_ids:
    cluster_id = df_leiden[df_leiden["node_id"] == id]["cluster_id"].values[0]
    all_cluster_ids = df_leiden[df_leiden['cluster_id'] == cluster_id]["node_id"].to_list()

    filtered_aux = df_aux[df_aux['node_id'].isin(all_cluster_ids)]
    filtered_aux = filtered_aux[filtered_aux["type"] == "agent"]

    average_pa_weight = filtered_aux.groupby("year")["pa_weight"].mean().reset_index()
    average_rec_weight = filtered_aux.groupby("year")["rec_weight"].mean().reset_index()
    average_fit_weight = filtered_aux.groupby("year")["fit_weight"].mean().reset_index()

    
    average_pa_weight.to_csv(f"avg_pa_cluster_{cluster_id}.csv", index=False)
    average_rec_weight.to_csv(f"avg_rec_cluster_{cluster_id}.csv", index=False)
    average_fit_weight.to_csv(f"avg_fit_cluster_{cluster_id}.csv", index=False)